In [2]:
import pandas as pd
df = pd.read_csv("diabetes_prediction_dataset.csv")
print(df.shape)
df.head()

(100000, 9)


,gender,age,hypertension,heart_disease,smoking_history,bmi,HbA1c_level,blood_glucose_level,diabetes
0,Female,80.0,0,1,never,25.19,6.6,140,0
1,Female,54.0,0,0,No Info,27.32,6.6,80,0
2,Male,28.0,0,0,never,27.32,5.7,158,0
3,Female,36.0,0,0,current,23.45,5.0,155,0
4,Male,76.0,1,1,current,20.14,4.8,155,0


In [3]:
print(df["diabetes"].value_counts())
print(df["diabetes"].value_counts(normalize=True))
df.isnull().sum()

diabetes
0    91500
1     8500
Name: count, dtype: int64
diabetes
0    0.915
1    0.085
Name: proportion, dtype: float64


gender                 0
age                    0
hypertension           0
heart_disease          0
smoking_history        0
bmi                    0
HbA1c_level            0
blood_glucose_level    0
diabetes               0
dtype: int64

In [4]:
X = pd.get_dummies(df.drop(columns="diabetes"), drop_first=False)
y = df["diabetes"]
print(X.shape)
X.head()

(100000, 15)


,age,hypertension,heart_disease,bmi,HbA1c_level,blood_glucose_level,gender_Female,gender_Male,gender_Other,smoking_history_No Info,smoking_history_current,smoking_history_ever,smoking_history_former,smoking_history_never,smoking_history_not current
0,80.0,0,1,25.19,6.6,140,True,False,False,False,False,False,False,True,False
1,54.0,0,0,27.32,6.6,80,True,False,False,True,False,False,False,False,False
2,28.0,0,0,27.32,5.7,158,False,True,False,False,False,False,False,True,False
3,36.0,0,0,23.45,5.0,155,True,False,False,False,True,False,False,False,False
4,76.0,1,1,20.14,4.8,155,False,True,False,False,True,False,False,False,False


In [5]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(X_train.shape, X_test.shape)

(80000, 15) (20000, 15)


In [6]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import confusion_matrix, classification_report

tree = DecisionTreeClassifier(criterion="entropy", max_depth=3, random_state=42)
tree.fit(X_train, y_train)

pred = tree.predict(X_test)
print("Train accuracy:", tree.score(X_train, y_train))
print("Test accuracy:", tree.score(X_test, y_test))
print(confusion_matrix(y_test, pred))
print(classification_report(y_test, pred))

Train accuracy: 0.9717625
Test accuracy: 0.9723
[[18300     0]
 [  554  1146]]
              precision    recall  f1-score   support

           0       0.97      1.00      0.99     18300
           1       1.00      0.67      0.81      1700

    accuracy                           0.97     20000
   macro avg       0.99      0.84      0.90     20000
weighted avg       0.97      0.97      0.97     20000



In [7]:
from sklearn.tree import export_text
print(export_text(tree, feature_names=list(X.columns)))

|--- HbA1c_level <= 6.70
|   |--- blood_glucose_level <= 210.00
|   |   |--- HbA1c_level <= 5.35
|   |   |   |--- class: 0
|   |   |--- HbA1c_level >  5.35
|   |   |   |--- class: 0
|   |--- blood_glucose_level >  210.00
|   |   |--- class: 1
|--- HbA1c_level >  6.70
|   |--- class: 1



In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv("diabetes_prediction_dataset.csv")

# stratified sample of 5000 rows so the from-scratch tree runs fast
sample, _ = train_test_split(df, train_size=5000, stratify=df["diabetes"], random_state=42)
sample = sample.copy()

# bin numeric columns into categories
sample["age"] = pd.cut(sample["age"], [0, 18, 40, 60, 120],
                       labels=["teen_or_child", "young_adult", "middle_age", "senior"]).astype(str)
sample["bmi"] = pd.cut(sample["bmi"], [0, 18.5, 25, 30, 100],
                       labels=["underweight", "normal", "overweight", "obese"]).astype(str)
sample["HbA1c_level"] = pd.cut(sample["HbA1c_level"], [0, 5.7, 6.5, 20],
                               labels=["low", "mid", "high"]).astype(str)
sample["blood_glucose_level"] = pd.cut(sample["blood_glucose_level"], [0, 100, 140, 200, 1000],
                                       labels=["low", "normal", "high", "very_high"]).astype(str)

# hypertension and heart_disease are 0/1, so treat them as categories too
sample["hypertension"] = sample["hypertension"].astype(str)
sample["heart_disease"] = sample["heart_disease"].astype(str)

train, test = train_test_split(sample, test_size=0.2, random_state=42, stratify=sample["diabetes"])
print(train.shape, test.shape)
train.head()

(4000, 9) (1000, 9)


,gender,age,hypertension,heart_disease,smoking_history,bmi,HbA1c_level,blood_glucose_level,diabetes
89745,Female,young_adult,0,0,No Info,obese,low,high,0
96534,Female,middle_age,0,0,never,overweight,high,very_high,1
23802,Female,teen_or_child,0,0,No Info,underweight,low,low,0
20749,Female,young_adult,0,0,never,overweight,mid,low,0
43689,Female,young_adult,0,0,current,underweight,mid,normal,0


In [2]:
def entropy(y):
    probs = y.value_counts(normalize=True)
    return -(probs * np.log2(probs)).sum()

# sanity check: 2 ones and 2 zeros should give exactly 1.0
print(entropy(pd.Series([1, 1, 0, 0])))
# pure node should give 0 (you may see -0.0, that's fine)
print(entropy(pd.Series([1, 1, 1, 1])))

1.0
-0.0


In [3]:
def information_gain(data, feature, target="diabetes"):
    total = entropy(data[target])
    weighted = 0
    for value, subset in data.groupby(feature):
        weighted += (len(subset) / len(data)) * entropy(subset[target])
    return total - weighted

features = [c for c in train.columns if c != "diabetes"]
print("Root entropy:", entropy(train["diabetes"]))
for f in features:
    print(f"{f:22s} gain = {information_gain(train, f):.4f}")

Root entropy: 0.41955649623854696
gender                 gain = 0.0025
age                    gain = 0.0564
hypertension           gain = 0.0165
heart_disease          gain = 0.0168
smoking_history        gain = 0.0159
bmi                    gain = 0.0348
HbA1c_level            gain = 0.0884
blood_glucose_level    gain = 0.1601


In [4]:
def id3(data, features, target="diabetes"):
    labels = data[target]

    # pure node -> leaf
    if labels.nunique() == 1:
        return labels.iloc[0]

    # no features left -> majority label
    if len(features) == 0:
        return labels.mode()[0]

    # choose the feature with the highest information gain
    gains = {f: information_gain(data, f, target) for f in features}
    best = max(gains, key=gains.get)

    tree = {best: {}}
    remaining = [f for f in features if f != best]

    # one branch per value of the chosen feature
    for value, subset in data.groupby(best):
        tree[best][value] = id3(subset, remaining, target)

    # fallback for values never seen during training
    tree[best]["_default"] = labels.mode()[0]
    return tree

In [5]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

def predict_one(tree, row):
    if not isinstance(tree, dict):      # reached a leaf
        return tree
    feature = next(iter(tree))
    branches = tree[feature]
    value = row[feature]
    if value in branches and value != "_default":
        return predict_one(branches[value], row)
    return branches["_default"]

def print_tree(tree, indent=""):
    if not isinstance(tree, dict):
        print(indent + "-> class", tree)
        return
    feature = next(iter(tree))
    for value, subtree in tree[feature].items():
        if value == "_default":
            continue
        print(f"{indent}{feature} = {value}")
        print_tree(subtree, indent + "    ")

id3_tree = id3(train, features)
print_tree(id3_tree)

preds = test.apply(lambda row: predict_one(id3_tree, row), axis=1)
print("Accuracy:", accuracy_score(test["diabetes"], preds))
print(confusion_matrix(test["diabetes"], preds))
print(classification_report(test["diabetes"], preds))

blood_glucose_level = high
    HbA1c_level = high
        age = middle_age
            bmi = normal
                hypertension = 0
                    smoking_history = No Info
                        -> class 0
                    smoking_history = current
                        -> class 0
                    smoking_history = ever
                        -> class 0
                    smoking_history = never
                        gender = Female
                            heart_disease = 0
                                -> class 0
                        gender = Male
                            -> class 0
                hypertension = 1
                    -> class 1
            bmi = obese
                smoking_history = No Info
                    gender = Female
                        hypertension = 0
                            heart_disease = 0
                                -> class 0
                    gender = Male
                        -> class 1
            

In [6]:
def count_leaves(tree):
    if not isinstance(tree, dict):
        return 1
    feature = next(iter(tree))
    return sum(count_leaves(sub) for v, sub in tree[feature].items() if v != "_default")

def depth(tree):
    if not isinstance(tree, dict):
        return 0
    feature = next(iter(tree))
    return 1 + max(depth(sub) for v, sub in tree[feature].items() if v != "_default")

print("Leaves:", count_leaves(id3_tree), "| Depth:", depth(id3_tree))

# train vs test accuracy to confirm overfitting
train_preds = train.apply(lambda r: predict_one(id3_tree, r), axis=1)
print("Train accuracy:", accuracy_score(train["diabetes"], train_preds))
print("Test accuracy:", accuracy_score(test["diabetes"], preds))

Leaves: 315 | Depth: 8
Train accuracy: 0.975
Test accuracy: 0.946


C4.5 ALGORITHM 


In [7]:
raw = pd.read_csv("diabetes_prediction_dataset.csv")
raw_sample, _ = train_test_split(raw, train_size=5000, stratify=raw["diabetes"], random_state=42)
train_c, test_c = train_test_split(raw_sample, test_size=0.2, random_state=42, stratify=raw_sample["diabetes"])
train_c, test_c = train_c.copy(), test_c.copy()

for d in (train_c, test_c):
    d["hypertension"] = d["hypertension"].astype(str)
    d["heart_disease"] = d["heart_disease"].astype(str)

numeric = ["age", "bmi", "HbA1c_level", "blood_glucose_level"]
features_c = [c for c in train_c.columns if c != "diabetes"]

In [8]:
def entropy_np(y):
    _, counts = np.unique(y, return_counts=True)
    p = counts / counts.sum()
    return -(p * np.log2(p)).sum()

def split_info(sizes):
    p = np.array(sizes) / np.sum(sizes)
    return -(p * np.log2(p)).sum()

def H(pos, n):
    # binary entropy, vectorized
    p = pos / n
    with np.errstate(divide="ignore", invalid="ignore"):
        h = -(p * np.log2(p) + (1 - p) * np.log2(1 - p))
    return np.nan_to_num(h)

def best_threshold(x, y):
    """Best binary split x <= t. Returns (gain, gain_ratio, threshold)."""
    order = np.argsort(x)
    xs, ys = x[order], y[order]
    n = len(ys)
    base = entropy_np(ys)
    pos_left = np.cumsum(ys)[:-1]
    n_left = np.arange(1, n)
    pos_right = ys.sum() - pos_left
    n_right = n - n_left
    valid = xs[1:] != xs[:-1]
    if not valid.any():
        return 0.0, 0.0, None
    gain = base - (n_left / n * H(pos_left, n_left) + n_right / n * H(pos_right, n_right))
    gain = np.where(valid, gain, -np.inf)
    i = int(np.argmax(gain))
    t = (xs[i] + xs[i + 1]) / 2
    si = H(n_left[i], n)
    return gain[i], (gain[i] / si if si > 0 else 0.0), t

def cat_gain_ratio(data, f):
    y = data["diabetes"].values
    n = len(y)
    weighted, sizes = 0, []
    for v, sub in data.groupby(f):
        weighted += len(sub) / n * entropy_np(sub["diabetes"].values)
        sizes.append(len(sub))
    gain = entropy_np(y) - weighted
    si = split_info(sizes)
    return gain, (gain / si if si > 0 else 0.0)

In [ ]:
def c45(data, features, depth=0, max_depth=5, min_samples=30):
    y = data["diabetes"].values
    majority = int(np.bincount(y).argmax())
    if len(np.unique(y)) == 1 or depth >= max_depth or len(data) < min_samples or not features:
        return {"leaf": majority}

    stats = {}
    for f in features:
        if f in numeric:
            stats[f] = best_threshold(data[f].values.astype(float), y)
        else:
            g, gr = cat_gain_ratio(data, f)
            stats[f] = (g, gr, None)

    avg_gain = np.mean([s[0] for s in stats.values()])
    cands = {f: s for f, s in stats.items() if s[0] >= avg_gain and s[0] > 1e-9}
    if not cands:
        return {"leaf": majority}
    best = max(cands, key=lambda f: cands[f][1])      # highest gain ratio

    if best in numeric:
        t = stats[best][2]
        left = data[data[best] <= t]
        right = data[data[best] > t]
        return {"feature": best, "type": "num", "threshold": t, "default": majority,
                "left": c45(left, features, depth + 1, max_depth, min_samples),
                "right": c45(right, features, depth + 1, max_depth, min_samples)}
    else:
        remaining = [f for f in features if f != best]
        branches = {v: c45(sub, remaining, depth + 1, max_depth, min_samples)
                    for v, sub in data.groupby(best)}
        return {"feature": best, "type": "cat", "branches": branches, "default": majority}

def predict_c45(tree, row):
    while "leaf" not in tree:
        f = tree["feature"]
        if tree["type"] == "num":
            tree = tree["left"] if row[f] <= tree["threshold"] else tree["right"]
        else:
            tree = tree["branches"].get(row[f], {"leaf": tree["default"]})
    return tree["leaf"]

def print_c45(tree, indent=""):
    if "leaf" in tree:
        print(indent + "-> class", tree["leaf"]); return
    f = tree["feature"]
    if tree["type"] == "num":
        print(f"{indent}{f} <= {tree['threshold']:.2f}"); print_c45(tree["left"], indent + "    ")
        print(f"{indent}{f} >  {tree['threshold']:.2f}"); print_c45(tree["right"], indent + "    ")
    else:
        for v, sub in tree["branches"].items():
            print(f"{indent}{f} = {v}"); print_c45(sub, indent + "    ")

c45_tree = c45(train_c, features_c)
print_c45(c45_tree)

p_train = train_c.apply(lambda r: predict_c45(c45_tree, r), axis=1)
p_test = test_c.apply(lambda r: predict_c45(c45_tree, r), axis=1)
print("Train accuracy:", accuracy_score(train_c["diabetes"], p_train))
print("Test accuracy:", accuracy_score(test_c["diabetes"], p_test))
print(confusion_matrix(test_c["diabetes"], p_test))
print(classification_report(test_c["diabetes"], p_test))